## Objectif du notebook

Ce notebook prépare les données nécessaires au volet B Data Analyst du projet Néovolt Grid+.

Les traitements réalisés sont :
- chargement des fichiers CSV ;
- conversion des dates ;
- conversion des variables numériques ;
- traitement des doublons ;
- identification des valeurs manquantes ;
- identification des valeurs négatives ou aberrantes ;
- création de variables temporelles ;
- enrichissement des relevés avec les compteurs, les clients et la météo ;
- production de fichiers nettoyés dans `data_cleaned/

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
ROOT_DIR = Path.cwd()

if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parents[1]

DATA_DIR = ROOT_DIR / "donnees"
OUTPUT_DIR = ROOT_DIR / "volet-b-data-analyst" / "data_cleaned"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR, OUTPUT_DIR

(WindowsPath('c:/Users/Master/Desktop/Mohamed/1CPDA/Examen_S2/ExaS2/neovolt-grid-plus/donnees'),
 WindowsPath('c:/Users/Master/Desktop/Mohamed/1CPDA/Examen_S2/ExaS2/neovolt-grid-plus/volet-b-data-analyst/data_cleaned'))

In [3]:
# Chargement des fichiers principaux du volet Data Analyst

releves = pd.read_csv(DATA_DIR / "releves_consommation.csv")
compteurs = pd.read_csv(DATA_DIR / "compteurs.csv")
clients = pd.read_csv(DATA_DIR / "clients.csv")
meteo = pd.read_csv(DATA_DIR / "meteo.csv")
incidents = pd.read_csv(DATA_DIR / "incidents_reseau.csv")
releves_horaires = pd.read_csv(DATA_DIR / "releves_horaires_echantillon.csv")
reclamations = pd.read_csv(DATA_DIR / "reclamations.csv")
fraudes = pd.read_csv(DATA_DIR / "cas_fraude_confirmes.csv")

print("releves :", releves.shape)
print("compteurs :", compteurs.shape)
print("clients :", clients.shape)
print("meteo :", meteo.shape)
print("incidents :", incidents.shape)
print("releves_horaires :", releves_horaires.shape)
print("reclamations :", reclamations.shape)
print("fraudes :", fraudes.shape)

releves : (512986, 4)
compteurs : (700, 9)
clients : (700, 7)
meteo : (5848, 5)
incidents : (420, 7)
releves_horaires : (21600, 4)
reclamations : (3000, 6)
fraudes : (24, 4)


In [4]:
# Conversion des colonnes de dates

releves["date"] = pd.to_datetime(releves["date"], errors="coerce")
compteurs["date_pose"] = pd.to_datetime(compteurs["date_pose"], errors="coerce")
clients["date_entree"] = pd.to_datetime(clients["date_entree"], errors="coerce")
meteo["date"] = pd.to_datetime(meteo["date"], errors="coerce")
incidents["date_debut"] = pd.to_datetime(incidents["date_debut"], errors="coerce")
releves_horaires["horodatage"] = pd.to_datetime(releves_horaires["horodatage"], errors="coerce")
reclamations["date"] = pd.to_datetime(reclamations["date"], errors="coerce")
fraudes["date_detection"] = pd.to_datetime(fraudes["date_detection"], errors="coerce")

In [5]:
# Conversion des colonnes numériques

releves["consommation_kwh"] = pd.to_numeric(releves["consommation_kwh"], errors="coerce")
releves_horaires["consommation_kwh"] = pd.to_numeric(releves_horaires["consommation_kwh"], errors="coerce")

meteo["temp_moyenne_c"] = pd.to_numeric(meteo["temp_moyenne_c"], errors="coerce")
meteo["temp_min_c"] = pd.to_numeric(meteo["temp_min_c"], errors="coerce")
meteo["temp_max_c"] = pd.to_numeric(meteo["temp_max_c"], errors="coerce")

clients["nb_personnes_foyer"] = pd.to_numeric(clients["nb_personnes_foyer"], errors="coerce")
clients["surface_m2"] = pd.to_numeric(clients["surface_m2"], errors="coerce")

compteurs["puissance_souscrite_kva"] = pd.to_numeric(compteurs["puissance_souscrite_kva"], errors="coerce")

incidents["duree_minutes"] = pd.to_numeric(incidents["duree_minutes"], errors="coerce")
incidents["nb_pdl_impactes"] = pd.to_numeric(incidents["nb_pdl_impactes"], errors="coerce")

reclamations["satisfaction"] = pd.to_numeric(reclamations["satisfaction"], errors="coerce")

In [6]:
# Suppression des doublons métier

nb_releves_avant = len(releves)
releves = releves.drop_duplicates(subset=["id_pdl", "date"], keep="first")
nb_releves_apres = len(releves)

nb_meteo_avant = len(meteo)
meteo = meteo.drop_duplicates(subset=["date", "zone"], keep="first")
nb_meteo_apres = len(meteo)

nb_horaires_avant = len(releves_horaires)
releves_horaires = releves_horaires.drop_duplicates(subset=["id_pdl", "horodatage"], keep="first")
nb_horaires_apres = len(releves_horaires)

nb_clients_avant = len(clients)
clients = clients.drop_duplicates(subset=["id_client"], keep="first")
nb_clients_apres = len(clients)

nb_compteurs_avant = len(compteurs)
compteurs = compteurs.drop_duplicates(subset=["id_pdl"], keep="first")
nb_compteurs_apres = len(compteurs)

print("Doublons supprimés :")
print("releves :", nb_releves_avant - nb_releves_apres)
print("meteo :", nb_meteo_avant - nb_meteo_apres)
print("releves_horaires :", nb_horaires_avant - nb_horaires_apres)
print("clients :", nb_clients_avant - nb_clients_apres)
print("compteurs :", nb_compteurs_avant - nb_compteurs_apres)

Doublons supprimés :
releves : 1286
meteo : 0
releves_horaires : 0
clients : 0
compteurs : 0


In [7]:
# Création de variables temporelles sur les relevés quotidiens

releves["annee"] = releves["date"].dt.year
releves["mois"] = releves["date"].dt.month
releves["jour"] = releves["date"].dt.day
releves["jour_semaine"] = releves["date"].dt.dayofweek
releves["nom_jour"] = releves["date"].dt.day_name()
releves["weekend"] = releves["jour_semaine"].isin([5, 6])

def definir_saison(mois):
    if mois in [12, 1, 2]:
        return "hiver"
    elif mois in [3, 4, 5]:
        return "printemps"
    elif mois in [6, 7, 8]:
        return "ete"
    elif mois in [9, 10, 11]:
        return "automne"
    return np.nan

releves["saison"] = releves["mois"].apply(definir_saison)

In [8]:
# Enrichissement des relevés avec les compteurs, les clients et la météo

conso_preparee = releves.merge(
    compteurs,
    on=["id_pdl", "zone"],
    how="left",
    suffixes=("", "_compteur")
)

conso_preparee = conso_preparee.merge(
    clients,
    on="id_client",
    how="left",
    suffixes=("", "_client")
)

conso_preparee = conso_preparee.merge(
    meteo,
    on=["date", "zone"],
    how="left"
)

print("Dataset préparé :", conso_preparee.shape)
conso_preparee.head()

Dataset préparé : (511700, 27)


,id_pdl,date,consommation_kwh,zone,annee,mois,jour,jour_semaine,nom_jour,weekend,...,statut,segment,commune,code_postal,date_entree,nb_personnes_foyer,surface_m2,temp_moyenne_c,temp_min_c,temp_max_c
0,PDL-000001,2024-01-01,18.14,Coteaux-Ouest,2024,1,1,0,Monday,False,...,actif,particulier,Coteaux-Ouest,69340,2015-02-02,2.0,60,1.51,-2.10,6.01
1,PDL-000001,2024-01-02,15.01,Coteaux-Ouest,2024,1,2,1,Tuesday,False,...,actif,particulier,Coteaux-Ouest,69340,2015-02-02,2.0,60,4.58,-0.44,10.73
2,PDL-000001,2024-01-03,12.31,Coteaux-Ouest,2024,1,3,2,Wednesday,False,...,actif,particulier,Coteaux-Ouest,69340,2015-02-02,2.0,60,8.33,4.52,11.63
3,PDL-000001,2024-01-04,NaN,Coteaux-Ouest,2024,1,4,3,Thursday,False,...,actif,particulier,Coteaux-Ouest,69340,2015-02-02,2.0,60,-1.60,-6.04,3.45
4,PDL-000001,2024-01-05,18.79,Coteaux-Ouest,2024,1,5,4,Friday,False,...,actif,particulier,Coteaux-Ouest,69340,2015-02-02,2.0,60,0.58,-6.37,6.88


In [9]:
# Création des indicateurs de qualité sur la consommation

conso_preparee["consommation_kwh_brute"] = conso_preparee["consommation_kwh"]

conso_preparee["flag_conso_manquante"] = conso_preparee["consommation_kwh"].isna()
conso_preparee["flag_conso_negative"] = conso_preparee["consommation_kwh"] < 0

# Détection des valeurs aberrantes par méthode IQR globale
q1 = conso_preparee["consommation_kwh"].quantile(0.25)
q3 = conso_preparee["consommation_kwh"].quantile(0.75)
iqr = q3 - q1
borne_haute = q3 + 1.5 * iqr

conso_preparee["flag_conso_aberrante_iqr"] = conso_preparee["consommation_kwh"] > borne_haute

print("Borne haute IQR :", round(borne_haute, 2))
print("Valeurs manquantes :", conso_preparee["flag_conso_manquante"].sum())
print("Valeurs négatives :", conso_preparee["flag_conso_negative"].sum())
print("Valeurs aberrantes IQR :", conso_preparee["flag_conso_aberrante_iqr"].sum())

Borne haute IQR : 99.02
Valeurs manquantes : 6838
Valeurs négatives : 1032
Valeurs aberrantes IQR : 54179


In [10]:
# Création d'une consommation nettoyée pour les analyses

conso_preparee["consommation_kwh_clean"] = conso_preparee["consommation_kwh"]

# Les consommations négatives et aberrantes sont mises à NaN avant imputation
mask_invalid = (
    conso_preparee["flag_conso_manquante"]
    | conso_preparee["flag_conso_negative"]
    | conso_preparee["flag_conso_aberrante_iqr"]
)

conso_preparee.loc[mask_invalid, "consommation_kwh_clean"] = np.nan

# Imputation prudente par médiane id_pdl + mois
median_pdl_mois = conso_preparee.groupby(["id_pdl", "mois"])["consommation_kwh_clean"].transform("median")
conso_preparee["consommation_kwh_clean"] = conso_preparee["consommation_kwh_clean"].fillna(median_pdl_mois)

# Si encore manquant, imputation par médiane zone + mois
median_zone_mois = conso_preparee.groupby(["zone", "mois"])["consommation_kwh_clean"].transform("median")
conso_preparee["consommation_kwh_clean"] = conso_preparee["consommation_kwh_clean"].fillna(median_zone_mois)

# Si encore manquant, imputation par médiane globale
median_globale = conso_preparee["consommation_kwh_clean"].median()
conso_preparee["consommation_kwh_clean"] = conso_preparee["consommation_kwh_clean"].fillna(median_globale)

conso_preparee["flag_conso_imputee"] = mask_invalid

print("Valeurs manquantes restantes dans consommation_kwh_clean :", conso_preparee["consommation_kwh_clean"].isna().sum())
print("Valeurs imputées :", conso_preparee["flag_conso_imputee"].sum())

Valeurs manquantes restantes dans consommation_kwh_clean : 0
Valeurs imputées : 62049


In [11]:
# Création de variables météo utiles pour l'analyse

# Degrés-jour de chauffage avec une température de référence de 17°C
conso_preparee["degres_jour_chauffage"] = np.maximum(0, 17 - conso_preparee["temp_moyenne_c"])

# Indicateur simple de journée froide
conso_preparee["jour_froid"] = conso_preparee["temp_moyenne_c"] < 7

# Indicateur simple de journée chaude
conso_preparee["jour_chaud"] = conso_preparee["temp_moyenne_c"] > 25

In [12]:
# Préparation des incidents réseau

incidents["date"] = incidents["date_debut"].dt.date
incidents["date"] = pd.to_datetime(incidents["date"], errors="coerce")
incidents["annee"] = incidents["date"].dt.year
incidents["mois"] = incidents["date"].dt.month

incidents_preparees = incidents.copy()

incidents_preparees.head()

,id_incident,date_debut,duree_minutes,zone,type,nb_pdl_impactes,cause,date,annee,mois
0,INC-0001,2024-09-19 09:56:00,91,Parc-Tertiaire,surtension,119,inconnue,2024-09-19,2024,9
1,INC-0002,2025-11-09 17:53:00,83,Zone-Industrielle,maintenance_programmee,44,inconnue,2025-11-09,2025,11
2,INC-0003,2024-05-02 10:52:00,28,Coteaux-Ouest,coupure,68,vetuste_materiel,2024-05-02,2024,5
3,INC-0004,2024-10-21 23:25:00,5,Zone-Industrielle,panne_poste,993,defaut_isolement,2024-10-21,2024,10
4,INC-0005,2025-07-26 03:11:00,35,Zone-Industrielle,surtension,693,inconnue,2025-07-26,2025,7


In [13]:
# Préparation des relevés horaires

releves_horaires_preparees = releves_horaires.copy()

releves_horaires_preparees["date"] = releves_horaires_preparees["horodatage"].dt.date
releves_horaires_preparees["date"] = pd.to_datetime(releves_horaires_preparees["date"], errors="coerce")
releves_horaires_preparees["heure"] = releves_horaires_preparees["horodatage"].dt.hour
releves_horaires_preparees["jour_semaine"] = releves_horaires_preparees["horodatage"].dt.dayofweek
releves_horaires_preparees["weekend"] = releves_horaires_preparees["jour_semaine"].isin([5, 6])

releves_horaires_preparees.head()

,id_pdl,horodatage,consommation_kwh,zone,date,heure,jour_semaine,weekend
0,PDL-000001,2025-08-23 00:00:00,0.12,Coteaux-Ouest,2025-08-23,0,5,True
1,PDL-000001,2025-08-23 01:00:00,0.09,Coteaux-Ouest,2025-08-23,1,5,True
2,PDL-000001,2025-08-23 02:00:00,0.10,Coteaux-Ouest,2025-08-23,2,5,True
3,PDL-000001,2025-08-23 03:00:00,0.08,Coteaux-Ouest,2025-08-23,3,5,True
4,PDL-000001,2025-08-23 04:00:00,0.10,Coteaux-Ouest,2025-08-23,4,5,True


In [14]:
# Préparation des réclamations clients

reclamations_preparees = reclamations.copy()

reclamations_preparees["annee"] = reclamations_preparees["date"].dt.year
reclamations_preparees["mois"] = reclamations_preparees["date"].dt.month
reclamations_preparees["longueur_texte"] = reclamations_preparees["texte"].astype(str).str.len()

reclamations_preparees.head()

,id_reclamation,id_client,date,canal,texte,satisfaction,annee,mois,longueur_texte
0,REC-00001,CLI-00649,2025-12-07,email,Je n'ai pas donne mon accord pour la transmiss...,1,2025,12,120
1,REC-00002,CLI-00541,2025-12-02,espace_client,Je tiens a remercier le technicien intervenu e...,5,2025,12,91
2,REC-00003,CLI-00064,2025-01-24,courrier,Les microcoupures se multiplient depuis janvie...,2,2025,1,105
3,REC-00004,CLI-00570,2024-06-23,courrier,"Vous m'avez preleve deux fois ce mois-ci, je v...",2,2024,6,91
4,REC-00005,CLI-00088,2026-02-19,espace_client,"Je conteste ma facture de fevrier, le montant ...",2,2026,2,142


In [15]:
# Export des fichiers préparés

conso_preparee.to_csv(OUTPUT_DIR / "consommation_preparee.csv", index=False, encoding="utf-8")
incidents_preparees.to_csv(OUTPUT_DIR / "incidents_preparees.csv", index=False, encoding="utf-8")
releves_horaires_preparees.to_csv(OUTPUT_DIR / "releves_horaires_preparees.csv", index=False, encoding="utf-8")
reclamations_preparees.to_csv(OUTPUT_DIR / "reclamations_preparees.csv", index=False, encoding="utf-8")
fraudes.to_csv(OUTPUT_DIR / "cas_fraude_confirmes_preparees.csv", index=False, encoding="utf-8")

print("Fichiers exportés dans :", OUTPUT_DIR)

Fichiers exportés dans : c:\Users\Master\Desktop\Mohamed\1CPDA\Examen_S2\ExaS2\neovolt-grid-plus\volet-b-data-analyst\data_cleaned


In [16]:
# Rapport de synthèse du nettoyage

rapport_nettoyage = pd.DataFrame([
    {
        "indicateur": "lignes_consommation_preparee",
        "valeur": len(conso_preparee)
    },
    {
        "indicateur": "consommations_manquantes_initiales",
        "valeur": int(conso_preparee["flag_conso_manquante"].sum())
    },
    {
        "indicateur": "consommations_negatives_detectees",
        "valeur": int(conso_preparee["flag_conso_negative"].sum())
    },
    {
        "indicateur": "consommations_aberrantes_iqr_detectees",
        "valeur": int(conso_preparee["flag_conso_aberrante_iqr"].sum())
    },
    {
        "indicateur": "consommations_imputees",
        "valeur": int(conso_preparee["flag_conso_imputee"].sum())
    },
    {
        "indicateur": "valeurs_manquantes_restantes_conso_clean",
        "valeur": int(conso_preparee["consommation_kwh_clean"].isna().sum())
    },
    {
        "indicateur": "borne_haute_iqr_consommation",
        "valeur": round(float(borne_haute), 2)
    }
])

rapport_nettoyage.to_csv(OUTPUT_DIR / "rapport_nettoyage.csv", index=False, encoding="utf-8")

rapport_nettoyage

,indicateur,valeur
0,lignes_consommation_preparee,511700.00
1,consommations_manquantes_initiales,6838.00
2,consommations_negatives_detectees,1032.00
3,consommations_aberrantes_iqr_detectees,54179.00
4,consommations_imputees,62049.00
5,valeurs_manquantes_restantes_conso_clean,0.00
6,borne_haute_iqr_consommation,99.03


## Conclusion du nettoyage

Le dataset principal `consommation_preparee.csv` a été généré à partir des relevés de consommation enrichis avec les informations compteurs, clients et météo.

Les anomalies de consommation ne sont pas supprimées sans trace :
- la consommation brute est conservée ;
- les valeurs manquantes, négatives et aberrantes sont identifiées par des flags ;
- une consommation nettoyée est créée pour les analyses ;
- les valeurs imputées sont marquées.

Cette approche permet de produire des analyses exploitables tout en gardant une traçabilité des limites de qualité des données.